# Lesson 20: Zava Retail Capstone (Part 1) — 4-Layer Architecture & RLS

In this notebook, you will:
1. Provision the multi-tenant **Zava Retail** SQLite database (`stores`, `customers`, `products`, `sales_transactions`).
2. Enforce store-scoped **Row-Level Security (RLS)** in Python's Data Access Layer.
3. Compare it against the production PostgreSQL `CREATE POLICY` DDL.

In [ ]:
import sqlite3

class ZavaDatabaseProvider:
    def __init__(self):
        self.conn = sqlite3.connect(":memory:")
        self.conn.row_factory = sqlite3.Row
        self._seed()

    def _seed(self):
        self.conn.executescript("""
        CREATE TABLE stores (store_id TEXT PRIMARY KEY, store_name TEXT, region TEXT);
        CREATE TABLE customers (customer_id INT PRIMARY KEY, store_id TEXT, name TEXT, loyalty_tier TEXT);
        CREATE TABLE products (sku TEXT PRIMARY KEY, store_id TEXT, name TEXT, category TEXT, price REAL);
        CREATE TABLE sales_transactions (tx_id INT PRIMARY KEY, store_id TEXT, customer_id INT, total REAL);
        INSERT INTO stores VALUES ('seattle','Zava Seattle','west'),('redmond','Zava Redmond','west'),('bellevue','Zava Bellevue','west'),('online','Zava Online','global');
        INSERT INTO customers VALUES (1,'seattle','Maya Lin','gold'),(2,'redmond','Noah Kim','silver'),(3,'online','Elena Cruz','platinum');
        INSERT INTO products VALUES ('SKU-1','seattle','Aero Trail Runner','Footwear',129.0),('SKU-2','seattle','StormShell Rain Jacket','Apparel',185.0),('SKU-3','redmond','Aero Trail Runner','Footwear',129.0);
        INSERT INTO sales_transactions VALUES (101,'seattle',1,314.0),(102,'redmond',2,129.0),(103,'online',3,450.0);
        """)

    def get_store_sales(self, rls_store_id: str) -> list[dict]:
        rows = self.conn.execute(
            "SELECT tx_id, store_id, total FROM sales_transactions WHERE store_id = ?",
            (rls_store_id,),
        ).fetchall()
        return [dict(r) for r in rows]

provider = ZavaDatabaseProvider()
print("Seattle RLS sales:", provider.get_store_sales("seattle"))
print("Redmond RLS sales:", provider.get_store_sales("redmond"))

### Challenge Exercise
Create a `SQLite` `VIEW` or helper `execute_scoped(sql, rls_store_id)` that automatically wraps queries against `customers` and `sales_transactions` in a CTE filtered by `store_id = ?`.